# VULCAN
## IMPORTING MODULES

In [87]:
import pandas as pd
import numpy as np

## LOADING RAW CSVs

In [88]:
complaints = pd.read_csv('complaints.csv')
customers = pd.read_csv('customers.csv')
service = pd.read_csv('service_records.csv')
vehicles = pd.read_csv('vehicles.csv')

print(complaints.shape, customers.shape, service.shape, vehicles.shape)

(350, 6) (415, 6) (2210, 7) (550, 8)


## INSPECTING THE CSVs

### complaints.csv

In [89]:
complaints.head()

,complaint_id,vehicle_id,complaint_date,category,status,resolution_days
0,CMP0001,V0137,2025-04-19,Battery Drain,Escalated,3.0
1,CMP0002,V0062,2026-03-29,Noise/Vibration,open,0.0
2,CMP0003,V0273,2026-08-06,Billing Issue,Resolved,0.0
3,CMP0004,V0328,2025-02-06,Poor Staff Behaviour,Resolved,14.0
4,CMP0005,V0194,2025-04-09,Poor Staff Behaviour,Resolved,3.0


In [90]:
complaints.describe()

,resolution_days
count,287.000000
mean,5.034843
std,4.753229
min,0.000000
25%,2.000000
50%,3.000000
75%,8.000000
max,14.000000


In [91]:
complaints.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   complaint_id     350 non-null    object 
 1   vehicle_id       350 non-null    object 
 2   complaint_date   350 non-null    object 
 3   category         342 non-null    object 
 4   status           350 non-null    object 
 5   resolution_days  287 non-null    float64
dtypes: float64(1), object(5)
memory usage: 16.5+ KB


In [92]:
complaints.isna().sum()

complaint_id        0
vehicle_id          0
complaint_date      0
category            8
status              0
resolution_days    63
dtype: int64

In [93]:
complaints['category'].value_counts()

category
Billing Issue           44
Poor Staff Behaviour    38
Charging Issue          38
Delayed Service         38
Battery Drain           37
Part Unavailability     36
Brake Issue             33
AC Not Cooling          31
Noise/Vibration         25
Software Glitch         22
Name: count, dtype: int64

No obvious category inconsistency was observed from the displayed values.

In [94]:
complaints['status'].value_counts()

status
Resolved     64
Open         63
RESOLVED     61
Escalated    58
open         58
Closed       46
Name: count, dtype: int64

There are same categories that are being treated different just because they are written differently. We have to resolve this issue.

In [95]:
complaints_clean = complaints.copy()

complaints_clean['status'] = complaints_clean['status'].str.strip().str.capitalize()

In [96]:
complaints_clean['status'].value_counts()

status
Resolved     125
Open         121
Escalated     58
Closed        46
Name: count, dtype: int64

Now, we have the 4 standard status - Resolved, Open, Escalated, Closed

***Hypothesis-1***

From this data, we can make a hypothesis that the missing values in resolution_days may be missing because those complaints have not yet been resolved — meaning complaints with an unresolved status such as Open (including open) may have resolution_days = NaN.

In [97]:
complaints_clean[complaints_clean['resolution_days'].isna()]['status'].value_counts()

status
Open         25
Resolved     23
Escalated     9
Closed        6
Name: count, dtype: int64

The missing values contains the values from all the categories, so we reject this hypothesis.

Now lets move on to the next investigation. We want to know if every complaint ID is unique.

In [98]:
complaints_clean['complaint_id'].duplicated().sum()

np.int64(0)

Every complaint_id is unique. So there is no problem in this. 

Lets check if the dates are correct as they are being treated as objects right now. We want to know if they are consistantly formatted.

In [99]:
complaints_clean['complaint_date'].head(10)

0    2025-04-19
1    2026-03-29
2    2026-08-06
3    2025-02-06
4    2025-04-09
5    2026-08-02
6    2026-01-10
7    2025-07-13
8    2026-07-23
9    2025-06-17
Name: complaint_date, dtype: object

The dates appear to be in a conistant format of YYYY-MM-DD, but we need to check if it is true for the whole data. 

In [100]:
pd.to_datetime(complaints_clean['complaint_date'], errors= 'coerce').isna().sum()

np.int64(0)

The values are consistant throughout the dataset. So we convert the dates to the proper date format.

In [101]:
complaints_clean['complaint_date'] = pd.to_datetime(complaints_clean['complaint_date'])

In [102]:
complaints_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   complaint_id     350 non-null    object        
 1   vehicle_id       350 non-null    object        
 2   complaint_date   350 non-null    datetime64[ns]
 3   category         342 non-null    object        
 4   status           350 non-null    object        
 5   resolution_days  287 non-null    float64       
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 16.5+ KB


The complaint_date comlumn was succesfully converted to `datetime64` from `object`.

We have cleaned the complaint_date column and have standardized the status column. So we focus on the category column now, where 8 values are missing.

The question is - *What do these 8 missing complaints look like and can we find information in other columns regarding this.*

In [103]:
complaints_clean[complaints_clean['category'].isna()]

,complaint_id,vehicle_id,complaint_date,category,status,resolution_days
74,CMP0075,V0069,2026-04-13,NaN,Open,0.0
80,CMP0081,V0252,2025-07-19,NaN,Escalated,2.0
173,CMP0174,V0217,2025-07-31,NaN,Escalated,NaN
259,CMP0260,V0206,2025-01-22,NaN,Closed,2.0
301,CMP0302,V0426,2026-04-21,NaN,Resolved,NaN
323,CMP0324,V0085,2026-03-02,NaN,Closed,3.0
342,CMP0343,V0049,2026-04-26,NaN,Escalated,14.0
347,CMP0348,V0066,2025-08-17,NaN,Resolved,1.0


From this data we were not able to find any relation between category and status. Nor were we able to guess a value from the other columns. So we will label them *Uncategorized*. 

In [104]:
complaints_clean['category'] = complaints_clean['category'].fillna('Uncategorized')

In [105]:
complaints_clean['category'].isna().sum()

np.int64(0)

Lets move on from the category column to the main unresolved issue - the *resolution_days* column. 

In [106]:
complaints_clean[complaints_clean['resolution_days'].isna()]

,complaint_id,vehicle_id,complaint_date,category,status,resolution_days
29,CMP0030,V0178,2025-06-16,Billing Issue,Resolved,NaN
49,CMP0050,V0128,2025-09-12,Brake Issue,Resolved,NaN
51,CMP0052,V0287,2025-06-26,Delayed Service,Resolved,NaN
55,CMP0056,V0390,2025-03-24,Billing Issue,Resolved,NaN
70,CMP0071,V0163,2026-01-15,Part Unavailability,Open,NaN
...,...,...,...,...,...,...
324,CMP0325,V0381,2026-01-26,Charging Issue,Open,NaN
333,CMP0334,V0350,2024-12-05,Part Unavailability,Escalated,NaN
334,CMP0335,V0543,2025-11-25,Brake Issue,Open,NaN
343,CMP0344,V0542,2025-09-08,Part Unavailability,Open,NaN


`resolution_days` contained 63 missing values. Investigation showed that these missing values were spread across various complaint status and categories. So we cannont actually inferr the values from these fields. The missing values were therefore retained rather than artificially imputed.

In [107]:
complaints_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   complaint_id     350 non-null    object        
 1   vehicle_id       350 non-null    object        
 2   complaint_date   350 non-null    datetime64[ns]
 3   category         350 non-null    object        
 4   status           350 non-null    object        
 5   resolution_days  287 non-null    float64       
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 16.5+ KB


The **complaints.csv** is cleaned for now. Now we will move to the customers.csv

### customers.csv

In [108]:
customers.head

<bound method NDFrame.head of     customer_id              name       phone                        email  \
0         C0001      Allison Hill  1960013389  hoffmanjennifer@example.net   
1         C0002     Kevin Pacheco  4235116155        lindsay78@example.org   
2         C0003     Juan Calderon   341316475    mitchellclark@example.com   
3         C0004       Andrea Reid  4835030564      clarksherri@example.net   
4         C0005    Jennifer Rocha  3884969653     williamdavis@example.org   
..          ...               ...         ...                          ...   
410       C9010     PATRICK HARDY  5346554947     ruizmichelle@example.net   
411       C9011   STEPHEN ALVAREZ  6988649767          adennis@example.org   
412       C9012    Matthew Smith   5102726149      jacobconrad@example.com   
413       C9013        CATHY BELL  5528938911     ricejennifer@example.com   
414       C9014  Jennifer Mccall   8338988816            hmoon@example.org   

           city signup_date  
0  

In [109]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415 entries, 0 to 414
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  415 non-null    object
 1   name         415 non-null    object
 2   phone        415 non-null    int64 
 3   email        389 non-null    object
 4   city         415 non-null    object
 5   signup_date  415 non-null    object
dtypes: int64(1), object(5)
memory usage: 19.6+ KB


We can see that the `email` column has values missing. Lets see the exact number of missing values.

In [110]:
customers.isna().sum()

customer_id     0
name            0
phone           0
email          26
city            0
signup_date     0
dtype: int64

There are 26 missing values in the `email` column. We will inspect the missing values and try to find a relation between other columns to see if we can reasonably fill the missing values.

In [111]:
customers_clean = customers.copy()

In [112]:
customers_clean[customers_clean['email'].isna()]

,customer_id,name,phone,email,city,signup_date
5,C0006,Christopher Becker,7848018451,NaN,Gurugram,2023-10-16
20,C0021,Joseph Drake,5951484656,NaN,Ghaziabad,2025-05-16
24,C0025,Victoria Larson,8727743487,NaN,Faridabad,2025-02-04
91,C0092,Lauren Robles,9930972896,NaN,Mumbai,2023-03-22
93,C0094,Tyler Stevens,9733484344,NaN,Delhi,2025-05-27
106,C0107,Amber Cooper,6246287334,NaN,Jaipur,2025-12-31
126,C0127,Holly Gonzalez,5969691557,NaN,Jaipur,2025-01-21
127,C0128,Michele Evans,3412528669,NaN,Pune,2023-03-08
131,C0132,Michael Chen,3095849825,NaN,Gurugram,2026-06-22
135,C0136,Bradley Beck,3611037129,NaN,Gurugram,2023-07-12


26 customer records have missing email addresses. After examining the affected records, the missing emails could not be reliably inferred from the available columns such as name, phone number, city, or signup date. Therefore, the missing values will not be fabricated and will be represented as `unknown`.

In [113]:
customers_clean['email'] = customers_clean['email'].fillna('unknown')

In [114]:
customers_clean.isna().sum()

customer_id    0
name           0
phone          0
email          0
city           0
signup_date    0
dtype: int64

There is no missing value in the data set now. Now we will check if all the values in the email column are actually in the right format or not. 

In [115]:
customers_clean.head(10)

,customer_id,name,phone,email,city,signup_date
0,C0001,Allison Hill,1960013389,hoffmanjennifer@example.net,Delhi,2024-10-10
1,C0002,Kevin Pacheco,4235116155,lindsay78@example.org,Faridabad,2026-06-14
2,C0003,Juan Calderon,341316475,mitchellclark@example.com,Gurugram,2025-05-22
3,C0004,Andrea Reid,4835030564,clarksherri@example.net,Mumbai,2023-11-15
4,C0005,Jennifer Rocha,3884969653,williamdavis@example.org,Lucknow,2026-01-05
5,C0006,Christopher Becker,7848018451,unknown,Gurugram,2023-10-16
6,C0007,Aaron Bowen,1489325288,georgetracy@example.org,Mumbai,2026-04-12
7,C0008,Sandra Parker,1718227824,ibrandt@example.net,Mumbai,2025-05-05
8,C0009,Jennifer Powers,1331509839,adrianzimmerman@example.org,Mumbai,2023-07-16
9,C0010,Shannon Jones,8299737631,hopkinsmichael@example.com,Chandigarh,2024-06-20


The emails look structurally consistant. The email column is cleaned now. Lets move to the `signup_date` column. 

In [117]:
customers_clean['signup_date'].head(10)

0    2024-10-10
1    2026-06-14
2    2025-05-22
3    2023-11-15
4    2026-01-05
5    2023-10-16
6    2026-04-12
7    2025-05-05
8    2023-07-16
9    2024-06-20
Name: signup_date, dtype: object

The `signup_date` column values are consistantly in the form of YYYY-MM-DD. Lets check if its true for the whole column.

In [118]:
pd.to_datetime(customers_clean['signup_date'], errors= 'coerce').isna().sum()

np.int64(0)

There is no unparseable value found in the investigation. So lets change the column in the proper date format.

In [120]:
customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'])

In [121]:
customers_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415 entries, 0 to 414
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   customer_id  415 non-null    object        
 1   name         415 non-null    object        
 2   phone        415 non-null    int64         
 3   email        415 non-null    object        
 4   city         415 non-null    object        
 5   signup_date  415 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 19.6+ KB


The data type of the *signup_date* columnn was changed from `object` to `datetime64`. 

The next column we check is the `phone` column. We need to know if all the phone numbers are right or are there anomalies in there.

In [122]:
customers_clean['phone'].astype(str).str.len().value_counts()

phone
10    379
9      33
8       3
Name: count, dtype: int64

After looking at the result we can say that - The 36 phone numbers that have less than 10 digits are incomplete phone numbers rather than intentionally shorter valid phone numbers. 



To test this hypothesis, we will check the rows with the incomplete phone numbers.

In [125]:
customers_clean[customers_clean['phone'].astype(str).str.len()<10]

,customer_id,name,phone,email,city,signup_date
2,C0003,Juan Calderon,341316475,mitchellclark@example.com,Gurugram,2025-05-22
19,C0020,Teresa Wilson,196556981,mlam@example.com,Lucknow,2022-10-25
26,C0027,Randy Shah,546688937,joshuatucker@example.com,Gurugram,2023-12-16
27,C0028,Jennifer Martin,699016272,greenmark@example.com,Chandigarh,2023-12-03
28,C0029,Jamie Walton,805310033,howelljohn@example.com,Jaipur,2025-04-06
41,C0042,Sharon Bryan,385977034,sarakim@example.net,Pune,2024-07-16
62,C0063,Daniel Taylor,762268388,charles06@example.org,Lucknow,2024-06-10
80,C0081,Chad Richardson,278681447,wendymitchell@example.com,Noida,2023-06-05
99,C0100,Jeremy Adams,453017426,hannah45@example.net,Faridabad,2023-07-04
104,C0105,Colin Golden,331190790,joymorales@example.org,Delhi,2024-10-02


We cannot tell if the numbers are correct or not. The next big question is wether some of the numbers are duplicated or not?

In [126]:
customers_clean['phone'].duplicated().sum()

np.int64(15)

Now we know that there are 36 phone numbers with less that 10 digits and 15 phone numbers that are duplicate. We will have a look at numbers that are duplicate before proceeding.

In [127]:
customers_clean[customers_clean['phone'].duplicated(keep=False)].sort_values('phone')

,customer_id,name,phone,email,city,signup_date
125,C0126,Crystal Sullivan,937966033,karen19@example.org,Mumbai,2025-10-09
401,C9001,CRYSTAL SULLIVAN,937966033,karen19@example.org,Mumbai,2025-10-09
408,C9008,CLAUDIA LYONS,1241904966,charlesvaughn@example.net,Jaipur,2023-08-05
29,C0030,Claudia Lyons,1241904966,charlesvaughn@example.net,Jaipur,2023-08-05
172,C0173,Sara Hoffman,1545444328,racheltrujillo@example.net,Delhi,2024-03-22
404,C9004,SARA HOFFMAN,1545444328,racheltrujillo@example.net,Delhi,2024-03-22
403,C9003,Jack Sandoval,1981898471,carolmejia@example.com,Lucknow,2023-12-06
339,C0340,Jack Sandoval,1981898471,carolmejia@example.com,Lucknow,2023-12-06
284,C0285,Nicole Acosta,2904904712,yorkjoshua@example.org,Mumbai,2025-01-19
409,C9009,NICOLE ACOSTA,2904904712,yorkjoshua@example.org,Mumbai,2025-01-19


Here we can see that there are many duplicate phone numbers that are caused due to duplicate records. Before deleting the duplicates, I want to see that - How many records are duplicate when we compare all the indentifying customer information?

In [129]:
customers_clean.duplicated(subset= ['name','phone','email','city','signup_date']).sum()

np.int64(0)

In [131]:
customers_clean[
    customers_clean['name'].str.lower().duplicated(keep=False)
].sort_values('name')

,customer_id,name,phone,email,city,signup_date
406,C9006,ASHLEY SCOTT,8877043581,jacksonstephanie@example.net,Ghaziabad,2024-09-23
197,C0198,Ashley Scott,8877043581,jacksonstephanie@example.net,Ghaziabad,2024-09-23
413,C9013,CATHY BELL,5528938911,ricejennifer@example.com,Noida,2023-04-25
408,C9008,CLAUDIA LYONS,1241904966,charlesvaughn@example.net,Jaipur,2023-08-05
401,C9001,CRYSTAL SULLIVAN,937966033,karen19@example.org,Mumbai,2025-10-09
324,C0325,Cathy Bell,5528938911,ricejennifer@example.com,Noida,2023-04-25
29,C0030,Claudia Lyons,1241904966,charlesvaughn@example.net,Jaipur,2023-08-05
125,C0126,Crystal Sullivan,937966033,karen19@example.org,Mumbai,2025-10-09
400,C9000,DAVID SOTO,7317912868,victor72@example.com,Pune,2025-06-06
398,C0399,David Soto,7317912868,victor72@example.com,Pune,2025-06-06
